In [ ]:
!pip install transformers accelerate torch pillow

In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
image = Image.open("full.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Extract all text from this image"}
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False)

inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=200)

result = processor.batch_decode(output, skip_special_tokens=True)

print(result[0])

system
You are a helpful assistant.
user
Extract all text from this image



In [15]:
image = Image.open("full.png")

width, height = image.size
scoreboard = image.crop((0, int(height * 0.92), width, int(height * 0.97)))  # stops at 97% instead of 100%

scoreboard.save("scoreboard_crop.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": scoreboard},
            {"type": "text", "text": "Extract all text from this image"}
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False)
inputs = processor(text=[text], images=[scoreboard], return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=200)
result = processor.batch_decode(output, skip_special_tokens=True)
print(result[0])

system
You are a helpful assistant.
user
Extract all text from this image

answer: AUS 17-2 1st Inns Toss Aus Smith 0* (3) Khawaja 4 (12) ENG Broad 2-13 (4)


In [14]:
image = Image.open("full.png")

width, height = image.size
scoreboard = image.crop((0, int(height * 0.92), width, int(height * 0.97)))  # stops at 97% instead of 100%

scoreboard.save("scoreboard_crop.png")  # verify the crop

To delete the gpu cache , These weights gets loaded

In [8]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()